In [1]:
import cv2
import numpy as np
import os

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input


In [3]:
dataset_path = r"I:\Fall-25\CVPR_FINAL_ASSIGNMENT3\dataset"

X = []
Y = []
label_map = {}
current_label = 0

IMG_SIZE = 224  

for person_name in os.listdir(dataset_path):
    person_path = os.path.join(dataset_path, person_name)
    if not os.path.isdir(person_path):
        continue

    label_map[current_label] = person_name

    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)

        img = cv2.imread(img_path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = preprocess_input(img)   

        X.append(img)
        Y.append(current_label)

    current_label += 1

X = np.array(X)
Y = np.array(Y)

print("Classes:", label_map)
print("X shape:", X.shape)


Classes: {0: 'Dibbo', 1: 'Isti', 2: 'Maruf', 3: 'Yasin'}
X shape: (46, 224, 224, 3)


In [12]:
base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224,224,3)   
)
base_model.trainable = False

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),          
    layers.RandomRotation(0.1),               
    layers.RandomZoom(0.1),                  
])

num_classes = len(label_map)

model = keras.Sequential([
    data_augmentation,
    base_model,

    layers.BatchNormalization(),
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),

    layers.Dense(num_classes, activation='softmax')
])



model.summary()


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_7 (Sequential)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,049,571 (15.45 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 4,049,571 (15.45 MB)

In [13]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


history = model.fit(
    X,
    Y,
    epochs=11,
    batch_size=8,
    validation_split=0.2,
    shuffle=True
)



Epoch 1/11
5/5 ━━━━━━━━━━━━━━━━━━━━ 11s 857ms/step - accuracy: 0.3056 - loss: 2.5621 - val_accuracy: 0.5000 - val_loss: 1.1519
Epoch 2/11
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 265ms/step - accuracy: 0.2778 - loss: 2.1723 - val_accuracy: 0.6000 - val_loss: 1.1282
Epoch 3/11
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 243ms/step - accuracy: 0.3889 - loss: 1.7145 - val_accuracy: 0.6000 - val_loss: 1.1166
Epoch 4/11
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 266ms/step - accuracy: 0.3056 - loss: 1.8671 - val_accuracy: 0.8000 - val_loss: 1.0987
Epoch 5/11
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 274ms/step - accuracy: 0.5000 - loss: 1.7348 - val_accuracy: 0.8000 - val_loss: 1.0788
Epoch 6/11
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 264ms/step - accuracy: 0.5833 - loss: 1.1020 - val_accuracy: 0.8000 - val_loss: 1.0622
Epoch 7/11
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 244ms/step - accuracy: 0.5278 - loss: 0.9778 - val_accuracy: 0.8000 - val_loss: 1.0471
Epoch 8/11
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 230ms/step - accuracy: 0.7222 - loss: 0.8954 - val_accuracy: 0.8000 - val_loss

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

cap = cv2.VideoCapture(0)

CONFIDENCE_THRESHOLD = 0.2

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]
        face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
        face = preprocess_input(face)
        face = np.expand_dims(face, axis=0)

        pred = model.predict(face, verbose=0)
        confidence = np.max(pred)
        label = np.argmax(pred)

        if confidence < CONFIDENCE_THRESHOLD:
            name = "Unknown"
        else:
            name = label_map[label]

        cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
        cv2.putText(
            frame,
            f"{name} ({confidence:.2f})",
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,255,0),
            2
        )

    cv2.imshow("EfficientNet Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
